In [1]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Conv2D, MaxPooling2D,Flatten
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
data = pd.read_excel("data/ex_data.xlsx" , sheet_name=None)
print(data.keys())

dict_keys(['개요', 'B2_항목정의', 'B2_학습데이터', 'B2_관능평가', 'C5_항목정의', 'C5_학습데이터', 'C5_자재기준정보', 'C5_일별원천', 'D2_항목정의', 'D2_학습데이터', 'D2_LOT별PLC구간'])


In [3]:
c5_training_data = data['C5_학습데이터']
c5MaterialMaster = data['C5_자재기준정보']
c5DailyRawData = data['C5_일별원천']
d2_training_data = data['D2_학습데이터']
d2LotPlcInterval = data['D2_LOT별PLC구간']

## B2 모델링

#### B2 데이터 로드

In [4]:
b2_training_data = data['B2_학습데이터']
b2SensoryEvaluation = data['B2_관능평가']

#### 데이터 전 처리

In [5]:
print(list(b2_training_data.columns))

['sample_id', 'src_type', 'split', 'is_aug', 'aug_src_id', 'lot_no', 'work_order_no', 'mfg_dt', 'batch_kg', 'prod_qty_ea', 'recipe_cd', 'recipe_nm', 'recipe_ver', 'item_cd', 'mix_RM-1001(정제수)', 'mix_RM-1002(간장)', 'mix_RM-1003(설탕)', 'mix_RM-1004(물엿)', 'mix_RM-1005(정제염)', 'mix_RM-1006(참기름)', 'mix_RM-1007(마늘(다진마늘))', 'mix_RM-1008(고춧가루)', 'mix_RM-1009(식초)', 'mix_RM-1010(멸치액젓(멜젓))', 'mix_RM-1011(된장)', 'mix_RM-1012(변성전분)', 'mix_RM-1013(잔탄검)', 'mix_RM-1014(고추장)', 'mix_RM-1015(양파(다진양파))', 'mix_RM-1016(L-글루탐산나트륨)', 'heating_temp', 'heating_time_min', 'eq_run_status', 'env_temp_wh', 'env_temp_mix', 'env_humidity_mix', 'season', 'month', 'mat_moisture_pct', 'mat_salt_pct', 'receive_dt', 'raw_age_days', 'brix', 'salt_pct', 'ph', 'visc_cps', 'sens_spicy', 'sens_sweet', 'sens_salty', 'sens_sour', 'sens_umami', 'sens_viscosity', 'sens_flavor', 'sens_overall', 'sens_n', 'sens_overall_std']


In [6]:
print('레시피 배힙학습 데이터')
b2_train = pd.DataFrame(b2_training_data[['mix_RM-1001(정제수)', 'mix_RM-1002(간장)', 'mix_RM-1003(설탕)', 'mix_RM-1004(물엿)', 'mix_RM-1005(정제염)', 'mix_RM-1006(참기름)', 
                                         'mix_RM-1007(마늘(다진마늘))', 'mix_RM-1008(고춧가루)', 'mix_RM-1009(식초)', 'mix_RM-1010(멸치액젓(멜젓))', 'mix_RM-1011(된장)', 
                                         'mix_RM-1012(변성전분)', 'mix_RM-1013(잔탄검)', 'mix_RM-1014(고추장)', 'mix_RM-1015(양파(다진양파))', 'mix_RM-1016(L-글루탐산나트륨)',
                                         'heating_temp', 'heating_time_min','brix', 'salt_pct', 'ph', 'visc_cps','recipe_cd']])
b2_train.info()

레시피 배힙학습 데이터
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mix_RM-1001(정제수)        4500 non-null   float64
 1   mix_RM-1002(간장)         4500 non-null   float64
 2   mix_RM-1003(설탕)         4500 non-null   float64
 3   mix_RM-1004(물엿)         4500 non-null   float64
 4   mix_RM-1005(정제염)        4500 non-null   float64
 5   mix_RM-1006(참기름)        4500 non-null   float64
 6   mix_RM-1007(마늘(다진마늘))   4500 non-null   float64
 7   mix_RM-1008(고춧가루)       4500 non-null   float64
 8   mix_RM-1009(식초)         4500 non-null   float64
 9   mix_RM-1010(멸치액젓(멜젓))   4500 non-null   float64
 10  mix_RM-1011(된장)         4500 non-null   float64
 11  mix_RM-1012(변성전분)       4500 non-null   float64
 12  mix_RM-1013(잔탄검)        4500 non-null   float64
 13  mix_RM-1014(고추장)        4500 non-null   float64
 14  mix_RM-1015(양파(다진양파))   450

In [7]:
# b2_train.isnull().sum()

In [8]:
b2_train.keys()

Index(['mix_RM-1001(정제수)', 'mix_RM-1002(간장)', 'mix_RM-1003(설탕)',
       'mix_RM-1004(물엿)', 'mix_RM-1005(정제염)', 'mix_RM-1006(참기름)',
       'mix_RM-1007(마늘(다진마늘))', 'mix_RM-1008(고춧가루)', 'mix_RM-1009(식초)',
       'mix_RM-1010(멸치액젓(멜젓))', 'mix_RM-1011(된장)', 'mix_RM-1012(변성전분)',
       'mix_RM-1013(잔탄검)', 'mix_RM-1014(고추장)', 'mix_RM-1015(양파(다진양파))',
       'mix_RM-1016(L-글루탐산나트륨)', 'heating_temp', 'heating_time_min', 'brix',
       'salt_pct', 'ph', 'visc_cps', 'recipe_cd'],
      dtype='object')

In [9]:
#recipe_mapping = b2_training_data.groupby('recipe_cd')['visc_cps'].mean()

#b2_train.loc[b2_train['visc_cps'].isnull(), 'visc_cps'] = (
#    b2_train.loc[b2_train['visc_cps'].isnull(), 'recipe_cd']
#    .map(recipe_mapping)
#)

In [10]:
b2_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mix_RM-1001(정제수)        4500 non-null   float64
 1   mix_RM-1002(간장)         4500 non-null   float64
 2   mix_RM-1003(설탕)         4500 non-null   float64
 3   mix_RM-1004(물엿)         4500 non-null   float64
 4   mix_RM-1005(정제염)        4500 non-null   float64
 5   mix_RM-1006(참기름)        4500 non-null   float64
 6   mix_RM-1007(마늘(다진마늘))   4500 non-null   float64
 7   mix_RM-1008(고춧가루)       4500 non-null   float64
 8   mix_RM-1009(식초)         4500 non-null   float64
 9   mix_RM-1010(멸치액젓(멜젓))   4500 non-null   float64
 10  mix_RM-1011(된장)         4500 non-null   float64
 11  mix_RM-1012(변성전분)       4500 non-null   float64
 12  mix_RM-1013(잔탄검)        4500 non-null   float64
 13  mix_RM-1014(고추장)        4500 non-null   float64
 14  mix_RM-1015(양파(다진양파))   4500 non-null   

In [11]:
b2_train.drop('recipe_cd', axis=1, inplace=True)
b2_train.head(10)

,mix_RM-1001(정제수),mix_RM-1002(간장),mix_RM-1003(설탕),mix_RM-1004(물엿),mix_RM-1005(정제염),mix_RM-1006(참기름),mix_RM-1007(마늘(다진마늘)),mix_RM-1008(고춧가루),mix_RM-1009(식초),mix_RM-1010(멸치액젓(멜젓)),...,mix_RM-1013(잔탄검),mix_RM-1014(고추장),mix_RM-1015(양파(다진양파)),mix_RM-1016(L-글루탐산나트륨),heating_temp,heating_time_min,brix,salt_pct,ph,visc_cps
0,30.104,5.016,12.115,10.085,1.006,3.051,7.643,4.956,1.956,0.000,...,0.202,0.00,0.000,0.199,95.3,27.8,42.4,4.90,5.47,5234.0
1,32.230,24.813,10.100,7.834,1.506,4.958,6.013,2.044,6.020,0.000,...,0.149,0.00,2.031,0.300,86.9,24.0,32.4,5.84,4.79,NaN
2,34.063,6.492,3.472,2.553,1.196,1.015,5.557,2.757,0.000,0.000,...,0.128,0.00,3.648,0.172,94.9,41.6,31.3,7.67,5.52,3308.0
3,30.082,5.077,11.678,10.099,0.985,2.890,7.973,5.050,2.036,0.000,...,0.197,0.00,0.000,0.200,92.0,24.3,41.0,4.81,5.38,NaN
4,38.617,5.876,3.993,2.975,1.477,1.006,6.030,2.958,0.000,0.000,...,0.100,0.00,3.043,0.198,90.8,32.8,29.6,6.44,5.61,NaN
5,40.756,6.465,4.546,2.389,1.243,1.194,4.510,2.998,0.000,0.000,...,0.084,0.00,2.677,0.169,85.3,38.4,29.9,6.50,5.59,2951.0
6,38.784,5.937,3.966,2.963,1.508,1.012,5.987,3.071,0.000,0.000,...,0.100,0.00,2.954,0.204,89.8,25.6,29.6,6.52,5.62,NaN
7,41.419,6.085,12.630,7.690,0.000,0.000,2.127,1.370,2.790,0.000,...,0.248,22.99,2.388,0.262,82.7,29.9,37.5,2.89,5.11,2408.0
8,30.180,4.937,12.334,9.831,1.013,2.952,7.940,5.033,1.994,0.000,...,0.196,0.00,0.000,0.205,91.7,31.9,41.9,4.89,5.43,4711.0
9,37.726,3.035,6.179,5.070,0.508,0.000,7.018,3.999,3.935,29.785,...,0.300,0.00,0.000,0.000,89.5,27.1,26.9,8.18,5.13,3984.0


In [12]:
b2_train.dropna(inplace=True)
b2_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2789 entries, 0 to 4499
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mix_RM-1001(정제수)        2789 non-null   float64
 1   mix_RM-1002(간장)         2789 non-null   float64
 2   mix_RM-1003(설탕)         2789 non-null   float64
 3   mix_RM-1004(물엿)         2789 non-null   float64
 4   mix_RM-1005(정제염)        2789 non-null   float64
 5   mix_RM-1006(참기름)        2789 non-null   float64
 6   mix_RM-1007(마늘(다진마늘))   2789 non-null   float64
 7   mix_RM-1008(고춧가루)       2789 non-null   float64
 8   mix_RM-1009(식초)         2789 non-null   float64
 9   mix_RM-1010(멸치액젓(멜젓))   2789 non-null   float64
 10  mix_RM-1011(된장)         2789 non-null   float64
 11  mix_RM-1012(변성전분)       2789 non-null   float64
 12  mix_RM-1013(잔탄검)        2789 non-null   float64
 13  mix_RM-1014(고추장)        2789 non-null   float64
 14  mix_RM-1015(양파(다진양파))   2789 non-null   float

#### X,y 데이터 분리

In [13]:
X = pd.DataFrame(b2_train[['mix_RM-1001(정제수)', 'mix_RM-1002(간장)', 'mix_RM-1003(설탕)', 'mix_RM-1004(물엿)', 'mix_RM-1005(정제염)', 'mix_RM-1006(참기름)',
              'mix_RM-1007(마늘(다진마늘))', 'mix_RM-1008(고춧가루)', 'mix_RM-1009(식초)', 'mix_RM-1010(멸치액젓(멜젓))', 'mix_RM-1011(된장)', 
              'mix_RM-1012(변성전분)', 'mix_RM-1013(잔탄검)', 'mix_RM-1014(고추장)', 'mix_RM-1015(양파(다진양파))', 'mix_RM-1016(L-글루탐산나트륨)',
              'heating_temp', 'heating_time_min']])

y = pd.DataFrame(b2_train[['brix', 'salt_pct', 'ph', 'visc_cps']])

brix_y = pd.DataFrame(b2_train['brix'])
salt_y = pd.DataFrame(b2_train['salt_pct'])
ph_y = pd.DataFrame(b2_train['ph'])
visc_y = pd.DataFrame(b2_train['visc_cps'])

In [14]:
print(len(X.columns))

18


### 독립 출력 모델

#### Brix 모델

In [15]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,brix_y, test_size=0.3, random_state=42
)

X_val, b_X_test, y_val, b_y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [16]:
R_brix_model = RandomForestRegressor(n_estimators=30, random_state=42, max_depth=20)
R_brix_model.fit(X_train, y_train)

c:\Users\guswj\miniconda3\envs\sfactroy\lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor(max_depth=20, n_estimators=30, random_state=42)

In [17]:
y_val_pred = R_brix_model.predict(X_val)

print("검증 RMSE:", mean_squared_error(y_val, y_val_pred) ** 0.5)
print("검증 R²:", r2_score(y_val, y_val_pred))

검증 RMSE: 0.5304573434490721
검증 R²: 0.9940902121024315


#### salt_pct 모델

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,salt_y, test_size=0.3, random_state=42
)

X_val, s_X_test, y_val, s_y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [19]:
R_salt_model = RandomForestRegressor(n_estimators=30, random_state=42, max_depth=20)
R_salt_model.fit(X_train, y_train)

c:\Users\guswj\miniconda3\envs\sfactroy\lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor(max_depth=20, n_estimators=30, random_state=42)

In [20]:
y_val_pred = R_salt_model.predict(X_val)

print("검증 RMSE:", mean_squared_error(y_val, y_val_pred) ** 0.5)
print("검증 R²:", r2_score(y_val, y_val_pred))

검증 RMSE: 0.16791102005085987
검증 R²: 0.9901800812331831


##### ph 모델

In [21]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,ph_y, test_size=0.3, random_state=42
)

X_val, p_X_test, y_val, p_y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [22]:
R_ph_model = RandomForestRegressor(n_estimators=30, random_state=42, max_depth=20)
R_ph_model.fit(X_train, y_train)

c:\Users\guswj\miniconda3\envs\sfactroy\lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor(max_depth=20, n_estimators=30, random_state=42)

In [23]:
y_val_pred = R_ph_model.predict(X_val)

print("검증 RMSE:", mean_squared_error(y_val, y_val_pred) ** 0.5)
print("검증 R²:", r2_score(y_val, y_val_pred))

검증 RMSE: 0.02503695313905971
검증 R²: 0.9938565385887528


##### visc_y 모델

In [24]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,visc_y, test_size=0.3, random_state=42
)

X_val, v_X_test, y_val, v_y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [25]:
R_visc_model = RandomForestRegressor(n_estimators=30, random_state=42, max_depth=20)
R_visc_model.fit(X_train, y_train)

c:\Users\guswj\miniconda3\envs\sfactroy\lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor(max_depth=20, n_estimators=30, random_state=42)

In [26]:
y_val_pred = R_visc_model.predict(X_val)

print("검증 RMSE:", mean_squared_error(y_val, y_val_pred) ** 0.5)
print("검증 R²:", r2_score(y_val, y_val_pred))

검증 RMSE: 275.02750984896494
검증 R²: 0.9509859266763527


In [27]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

models = {
    "Brix": {
        "model": R_brix_model,
        "X_test": b_X_test,
        "y_test": b_y_test,
    },
    "점도": {
        "model": R_visc_model,
        "X_test": v_X_test,
        "y_test": v_y_test,
    },
    "염도": {
        "model": R_salt_model,
        "X_test": s_X_test,
        "y_test": s_y_test,
    },
    "pH": {
        "model": R_ph_model,
        "X_test": p_X_test,
        "y_test": p_y_test,
    },
}

rows = []

for name, item in models.items():
    model = item["model"]

    try:
        y_pred = model.predict(item["X_test"], verbose=0)
    except TypeError:
        y_pred = model.predict(item["X_test"])

    y_true = np.asarray(item["y_test"]).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    rows.append({
        "모델": name,
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²": r2_score(y_true, y_pred),
    })

results_df = pd.DataFrame(rows)
display(results_df.round(4))

,모델,MSE,RMSE,R²
0,Brix,0.5063,0.7116,0.9905
1,점도,75909.4781,275.5167,0.9530
2,염도,0.0310,0.1760,0.9894
3,pH,0.0006,0.0254,0.9936
